# **simple llm architecture**

## **Installing dependencies**

In [54]:
!pip install torch

## **Importations**

In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pickle

### **Importing hyper-parameters and hardware selection**

In [16]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size=32
batch_size=64
max_iters=1000
learning_rate=3e-4
eval_iters=100
n_embd = 384
n_head=4
n_layer=4
dropout=0.2

cuda


### **Importing data loader**

In [17]:
chars=''
with open('/content/雲形紋章.txt','r',encoding='utf-8') as f:
  text=f.read()
  chars = sorted(list(set(text)))
vocab_size=len(chars)

## **Tokenization**

In [18]:
string_to_int={ ch:i for i,ch in enumerate(chars) }
int_to_string={ i:ch for i,ch in enumerate(chars) }
encode=lambda s: [string_to_int[c] for c in s]
decode= lambda l: ''.join([int_to_string[i] for i in l])

data= torch.tensor(encode(text),dtype=torch.long)

## **Train and Val split**

In [19]:
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]


## **estimating batch function**

In [20]:
def get_batch(split):
  data=train_data if split == 'train' else val_data
  ix= torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+block_size+1]for i in ix])
  x,y=x.to(device), y.to(device)
  return x,y

## **Estimate loss**

In [21]:
@torch.no_grad()
def estimate_loss():
  out={}
  model.eval()
  for split in ['train','val']:
    losses= torch.zeros(eval_iters)
    for k in range (eval_iters):
      X,Y=get_batch(split)
      logits,loss = model(X,Y)
      losses[k]= loss.item()
    out[split]=losses.mean()
  model.train()
  return out

## **Core Model**

In [22]:
class Head(nn.Module):
  def __init__(self,head_size):
    super().__init__()
    self.key=nn.Linear(n_embd,head_size,bias=False)
    self.query=nn.Linear(n_embd,head_size,bias=False)
    self.vallue=nn.Linear(n_embd,head_size,bias=False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size))) #reduce computations

    self.dropout=nn.Dropout(dropout)

  def forward(self,x):
    B,T,C=x.shape
    k=self.key(x)
    q=self.query(x)
    wei=q @ k.transpose(-2,-1)*k.shape[-1]**0.5
    wei=wei.masked_fill(self.tril[:T,:T]==0,float('-inf')) #scaling
    wei=F.softmax(wei,dim=-1)
    wei=self.dropout(wei)

    v=self.vallue(x)
    out=wei @ v
    return out


### **Multi Head Attention block**

In [23]:
class MultiHeadAttention(nn.Module):
  def __init__(self,num_heads,head_size):
    super().__init__()
    self.heads=nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj=nn.Linear(head_size * num_heads,n_embd)
    self.dropout=nn.Dropout(dropout)

  def forward(self,x):
    out=torch.cat([h(x) for h in self.heads],dim=-1)
    out=self.dropout(self.proj(out))
    return out

### **Feel Forward Block**

In [24]:
class FeedForward(nn.Module):
  def __init__(self,n_embd):
    super().__init__()
    self.net=nn.Sequential(
        nn.Linear(n_embd,4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd,n_embd),
        nn.Dropout(dropout),
    )
  def forward(self,x):
    return self.net(x)

### **Transformer block for Computation**

In [25]:
class Block(nn.Module):
  def __init__(self,n_embd,n_head):
    super().__init__()
    head_size=n_embd//n_head
    self.sa=MultiHeadAttention(n_head,head_size)
    self.ffwd=FeedForward(n_embd)

    '''For post norm architecture'''
    self.ln1=nn.LayerNorm(n_embd)
    self.ln2=nn.LayerNorm(n_embd)

  def forward(self,x):
    y=self.sa(x)
    x=self.ln1(x+y)
    y=self.ffwd(x)
    x=self.ln2(x+y)
    return x

In [26]:
class GPTLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.token_embedding_table=nn.Embedding(vocab_size,n_embd)
    self.postion_embedding_table=nn.Embedding(block_size,n_embd)
    self.blocks=nn.Sequential(*[Block(n_embd,n_head) for _ in range (n_layer)])
    self.ln_f=nn.LayerNorm(n_embd)
    self.lm_head=nn.Linear(n_embd,vocab_size)

    self.apply(self.__init__weights)
  def __init__weights(self,module):
    if isinstance(module,nn.Linear):
      torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module,nn.Embedding):
      torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)

  def forward(self,index,targets=None):
    #logits=self.token_embedding_table(index)
    B,T=index.shape
    tok_emb=self.token_embedding_table(index)
    pos_emb=self.postion_embedding_table(torch.arange(T,device=device))
    x=tok_emb+pos_emb
    x=self.blocks(x)
    x=self.ln_f(x)
    logits=self.lm_head(x)

    if targets is None:
      loss=None
    else:
      B,T,C=logits.shape
      logits=logits.view(B*T,C)
      targets=targets.view(B*T)
      loss = F.cross_entropy(logits,targets)

    return logits,loss

  def generate (self,index,max_new_tokens):
    for _ in range(max_new_tokens):
      logits,loss=self.forward(index)
      logits=logits[:,-1,:]
      probs=F.softmax(logits,dim=-1)
      index_next=torch.multinomial(probs,num_samples=1)
      index=torch.cat((index,index_next),dim=1)
    return index

model=GPTLanguageModel(vocab_size)
print('Loading Model')
with open('/content/drive/MyDrive/model/akari.pkl','rb') as f:
  model=pickle.load(f)
print('Model Loaded...')
m=model.to(device)

Loading Model
Model Loaded...


## **Pytorch Optimizer**

In [27]:
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate)

for iter in range(max_iters):
  if iter % eval_iters ==0:
    losses = estimate_loss()
    print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

  xb,yb=get_batch('train')

  logits,loss=model.forward(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss.item())

step: 0, train loss: 2.476, val loss: 3.014
step: 100, train loss: 2.496, val loss: 3.032
step: 200, train loss: 2.488, val loss: 3.051
step: 300, train loss: 2.478, val loss: 3.043
step: 400, train loss: 2.452, val loss: 3.034
step: 500, train loss: 2.435, val loss: 3.036
step: 600, train loss: 2.421, val loss: 3.049
step: 700, train loss: 2.426, val loss: 3.055
step: 800, train loss: 2.409, val loss: 3.058
step: 900, train loss: 2.386, val loss: 3.072
2.5651984214782715


In [85]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
import os
path='/content/drive/MyDrive/model'
os.makedirs(path, exist_ok=True)

In [29]:
import pickle
with open(os.path.join(path,'akari.pkl'),'wb') as f:
  pickle.dump(m,f)

In [30]:
with open('/content/drive/MyDrive/model/akari.pkl','rb') as f:
  m=pickle.load(f)
prompt=input('Prompt:')
context= torch.tensor(encode(prompt),dtype=torch.long,device=device)
generated_chars=decode(m.generate(context.unsqueeze(0),max_new_tokens=20)[0].tolist())
print(f"generated output:{generated_chars}")

Prompt:こんにちは
generated output:こんにちは」が赤靴に捧げようっとお世ったせてくださ
